# CPS_SQL_02_Joins_and_Windows


## Initial Setup (run once)
Connect to DuckDB and create tables from the CSVs in `../datasets/`.



In [ ]:
%load_ext sql
%sql duckdb:///../practice.duckdb
%config SqlMagic.displaylimit = None


In [ ]:
%%sql
CREATE OR REPLACE TABLE employees AS SELECT * FROM read_csv_auto('../datasets/employee_data_large.csv');
CREATE OR REPLACE TABLE sales AS SELECT * FROM read_csv_auto('../datasets/sales_large.csv');
CREATE OR REPLACE TABLE students AS SELECT * FROM read_csv_auto('../datasets/student_data_large.csv');
CREATE OR REPLACE TABLE departments AS SELECT * FROM read_csv_auto('../datasets/departments.csv');
CREATE OR REPLACE TABLE customers AS SELECT * FROM read_csv_auto('../datasets/customers.csv');
CREATE OR REPLACE TABLE orders AS SELECT * FROM read_csv_auto('../datasets/orders.csv');


## Setup (run once after kernel restart)
Reconnect to DuckDB.


In [ ]:
%reload_ext sql
%sql duckdb:///../practice.duckdb
%config SqlMagic.displaylimit = None


---

## Section 1: JOIN Fundamentals (employees ↔ departments)


1. Join `employees` to `departments` to show each employee’s `name`, `department`, and the department `location`.


In [ ]:
%%sql
SELECT
    e.name,
    e.department,
    d.location
FROM employees e
JOIN departments d
    ON e.department = d.department


2. Count employees per department location (requires a JOIN). Return `location` and `employee_count`.


In [ ]:
%%sql
SELECT
    d.location,
    COUNT(*) AS employee_count
FROM employees e
JOIN departments d 
    ON e.department = d.department
GROUP BY d.location
ORDER BY employee_count DESC


3. Compute average salary by department location. Ignore NULL salaries.


In [ ]:
%%sql
SELECT
    d.location,
    ROUND(AVG(e.salary), 2) AS avg_salary
FROM employees e 
JOIN departments d 
    ON e.department = d.department
GROUP BY d.location
ORDER BY avg_salary DESC

4. List all departments (even if they have zero employees) and show `employee_count` for each.


In [ ]:
%%sql
SELECT
    d.department,
    COUNT(e.employee_id) AS employee_count
FROM departments d
LEFT JOIN employees e
    ON d.department = e.department
GROUP BY d.department
ORDER BY employee_count DESC

5. Identify departments with **no** employees.


In [ ]:
%%sql
SELECT 
    d.department
FROM departments d 
LEFT JOIN employees e 
    ON d.department = e.department
WHERE e.employee_id IS NULL

6. For each department, show: `department`, `annual_budget_usd`, and `total_payroll` (sum of salaries). Ignore NULL salaries.


In [ ]:
%%sql
SELECT
    d.department,
    d.annual_budget_usd,
    COALESCE(ROUND(SUM(e.salary), 2), 0) AS total_payroll
FROM departments d
LEFT JOIN employees e
    ON d.department = e.department
GROUP BY d.department, d.annual_budget_usd
ORDER BY total_payroll DESC  

7. Find the departments where `total_payroll` exceeds `annual_budget_usd` (JOIN + aggregation + HAVING).


In [ ]:
%%sql
SELECT
    d.department,
    d.annual_budget_usd,
    COALESCE(ROUND(SUM(e.salary), 2), 0) AS total_payroll
FROM departments d
LEFT JOIN employees e
    ON d.department = e.department
GROUP BY d.department, d.annual_budget_usd
HAVING COALESCE(ROUND(SUM(e.salary), 2), 0) > d.annual_budget_usd
ORDER BY total_payroll DESC  

---

## Section 2: Multi-table JOINs (orders ↔ customers ↔ sales)


8. Create an order-level view showing: `order_id`, `order_date`, `customer_name`, `product_name`, `quantity`, `unit_price_usd`, and `revenue_usd`.


In [ ]:
%%sql
SELECT
    o.order_id,
    o.order_date,
    c.customer_name,
    s.product_name,
    o.quantity,
    o.unit_price_usd,
    o.revenue_usd
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
JOIN sales s
    ON o.product_id = s.product_id
ORDER BY o.order_date, o.order_id  

9. Total revenue by customer. Return `customer_id`, `customer_name`, and `total_revenue`.


In [ ]:
%%sql
SELECT
    c.customer_id,
    c.customer_name,
    COALESCE(ROUND(SUM(o.revenue_usd), 2), 0) AS total_revenue
FROM customers c
LEFT JOIN orders o
    ON o.customer_id = c.customer_id
GROUP BY c.customer_id, c.customer_name
ORDER BY total_revenue DESC

10. Top 10 customers by total revenue (descending).


In [ ]:
%%sql
SELECT 
    c.customer_id,
    c.customer_name,
    ROUND(SUM(o.revenue_usd), 2) AS total_revenue
FROM orders o 
JOIN customers c 
    ON o.customer_id = c.customer_id
GROUP BY c.customer_id, c.customer_name
ORDER BY total_revenue DESC
LIMIT 10

11. Total revenue by region (using customers.region).


In [ ]:
%%sql
SELECT
    c.region,
    ROUND(SUM(o.revenue_usd), 2) AS total_revenue
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
GROUP BY c.region
ORDER BY total_revenue DESC

12. Total quantity sold by product. Return `product_id`, `product_name`, `total_qty`.


In [ ]:
%%sql
SELECT
    s.product_id,
    s.product_name,
    SUM(o.quantity) AS total_qty
FROM orders o 
JOIN sales s 
    ON o.product_id = s.product_id
GROUP BY s.product_id, s.product_name
ORDER BY total_qty DESC
    


13. Top 5 products by total revenue (quantity * unit_price_usd or use revenue_usd).


In [ ]:
%%sql
SELECT 
    s.product_id,
    s.product_name,
    ROUND(SUM(o.revenue_usd), 2) AS total_revenue
FROM orders o
JOIN sales s
    ON o.product_id = s.product_id
GROUP BY s.product_id, s.product_name
ORDER BY total_revenue DESC
LIMIT 5


14. Find customers who have **no orders** (LEFT JOIN customers → orders).


In [ ]:
%%sql
SELECT 
    c.customer_id,
    c.customer_name
FROM customers c
LEFT JOIN orders o 
    ON c.customer_id = o.customer_id
WHERE o.order_id IS NULL


15. Find orders whose `customer_id` is missing from `customers` (LEFT JOIN orders → customers).


In [ ]:
%%sql
SELECT
    o.order_id,
    o.customer_id,
    o.order_date
FROM orders o
LEFT JOIN customers c
    ON o.customer_id = c.customer_id
WHERE c.customer_id IS NULL

---

## Section 3: CTEs & Subqueries


16. Using a CTE, compute total revenue per customer, then return only customers whose total revenue is above the **overall average** customer revenue.


In [ ]:
%%sql
WITH revenue_by_customer AS (
    SELECT 
        c.customer_id,
        c.customer_name,
        SUM(o.revenue_usd) AS total_revenue
    FROM customers c 
    JOIN orders o 
        ON c.customer_id = o.customer_id
    GROUP BY c.customer_id, c.customer_name
),
avg_customer_revenue AS (
    SELECT AVG(total_revenue) AS avg_revenue
    FROM revenue_by_customer
)

SELECT
    r.customer_id,
    r.customer_name,
    ROUND(r.total_revenue, 2) AS total_revenue
FROM revenue_by_customer r
CROSS JOIN avg_customer_revenue a
WHERE r.total_revenue > a.avg_revenue
ORDER BY r.total_revenue DESC

17. Compute monthly revenue (use `DATE_TRUNC('month', order_date)`). Return `month` and `revenue`.


In [ ]:
%%sql
SELECT
    DATE_TRUNC('month', order_date) AS month,
    ROUND(SUM(revenue_usd), 2) AS revenue
FROM orders
GROUP BY month
ORDER BY month

18. For each product, compute its percent of total revenue (product_revenue / total_revenue).


In [ ]:
%%sql
WITH product_revenue AS (
    SELECT
        s.product_id,
        s.product_name,
        SUM(o.revenue_usd) AS total_revenue
    FROM orders o
    JOIN sales s
        ON o.product_id = s.product_id
    GROUP BY s.product_id, s.product_name
),
overall_revenue AS (
    SELECT SUM(total_revenue) AS grand_total
    FROM product_revenue
)

SELECT
    p.product_id,
    p.product_name,
    ROUND(p.total_revenue, 2) AS total_revenue,
    ROUND(100.0 * p.total_revenue / o.grand_total, 2) AS percent_of_total
FROM product_revenue p
CROSS JOIN overall_revenue o
ORDER BY percent_of_total DESC



---

## Section 4: Window Functions (ranking, running totals, deltas)


19. Rank customers by total revenue within each region. Return `region`, `customer_id`, `customer_name`, `total_revenue`, and `revenue_rank` (use `RANK()` or `DENSE_RANK()`).


In [ ]:
%%sql
WITH customer_revenue AS (
    SELECT
        c.customer_id,
        c.customer_name,
        c.region,
        SUM(o.revenue_usd) AS total_revenue
    FROM customers c
    JOIN orders o
        ON c.customer_id = o.customer_id
    GROUP BY c.customer_id, c.customer_name, c.region
)

SELECT
    region,
    customer_id,
    customer_name,
    ROUND(total_revenue, 2) AS total_revenue,
    RANK() OVER (
        PARTITION BY region
        ORDER BY total_revenue DESC
    ) AS revenue_rank
FROM customer_revenue
ORDER BY region, revenue_rank

20. For each customer, compute their running total revenue over time by `order_date` (use `SUM(...) OVER (PARTITION BY ... ORDER BY ...)`).


In [ ]:
%%sql
SELECT
    o.customer_id,
    c.customer_name,
    o.order_date,
    ROUND(o.revenue_usd, 2) AS order_revenue,
    ROUND(
        SUM(o.revenue_usd) OVER (
            PARTITION BY o.customer_id
            ORDER BY o.order_date, o.order_id
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ),
        2
    ) AS running_total_revenue
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
ORDER BY o.customer_id, o.order_date, o.order_id

21. For each customer, compute the previous order’s revenue and the change from previous (use `LAG`).


In [ ]:
%%sql
SELECT
    o.customer_id,
    c.customer_name,
    o.order_date,
    o.order_id,
    ROUND(o.revenue_usd, 2) AS current_revenue,

    ROUND(
        LAG(o.revenue_usd) OVER (
            PARTITION BY o.customer_id
            ORDER BY o.order_date, o.order_id
        ),
        2
    ) AS previous_revenue,

    ROUND(
        o.revenue_usd - LAG(o.revenue_usd) OVER (
            PARTITION BY o.customer_id
            ORDER BY o.order_date, o.order_id
        ),
        2
    ) AS revenue_change

FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
ORDER BY o.customer_id, o.order_date, o.order_id


22. Compute overall running total revenue over time (no partition).


In [ ]:
%%sql
SELECT
    o.order_date,
    o.order_id,
    ROUND(o.revenue_usd, 2) AS order_revenue,
    ROUND(
        SUM(o.revenue_usd) OVER (
            ORDER BY o.order_date, o.order_id
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ),
        2
    ) AS running_total_revenue
FROM orders o
ORDER BY o.order_date, o.order_id

23. For each month, find the top product by revenue (use a window function like `ROW_NUMBER()` partitioned by month).


In [ ]:
%%sql
WITH monthly_product_revenue AS (
    SELECT
        DATE_TRUNC('month', o.order_date) AS month,
        s.product_id,
        s.product_name,
        SUM(o.revenue_usd) AS product_revenue
    FROM orders o
    JOIN sales s
        ON o.product_id = s.product_id
    GROUP BY month, s.product_id, s.product_name
),
ranked AS (
    SELECT
        month,
        product_id,
        product_name,
        product_revenue,
        ROW_NUMBER() OVER (
            PARTITION BY month
            ORDER BY product_revenue DESC
        ) AS rn
    FROM monthly_product_revenue
)

SELECT
    month,
    product_id,
    product_name,
    ROUND(product_revenue, 2) AS product_revenue
FROM ranked
WHERE rn = 1
ORDER BY month

24. For each customer, label their orders as 1st, 2nd, 3rd… purchase (use `ROW_NUMBER()` over customer ordered by date).


In [ ]:
%%sql
SELECT
    o.customer_id,
    c.customer_name,
    o.order_id,
    o.order_date,
    ROUND(o.revenue_usd, 2) AS revenue_usd,
    ROW_NUMBER() OVER (
        PARTITION BY o.customer_id
        ORDER BY o.order_date, o.order_id
    ) AS purchase_number
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
ORDER BY o.customer_id, purchase_number

---

## Section 5: Data Quality Checks


25. Check whether `order_id` is unique. Return any `order_id` values that appear more than once.


In [ ]:
%%sql
SELECT
    order_id,
    COUNT(*) AS occurrence_count
FROM orders
GROUP BY order_id
HAVING COUNT(*) > 1
ORDER BY occurrence_count DESC, order_id

26. Find any orders with non-positive quantity (quantity <= 0).


In [ ]:
%%sql
SELECT *
FROM orders
WHERE quantity <= 0
ORDER BY order_date, order_id
LIMIT 50

27. Find any orders with non-positive unit price (unit_price_usd <= 0).


In [ ]:
%%sql
SELECT *
FROM orders
WHERE unit_price_usd <= 0
ORDER BY order_date, order_id

28. Validate revenue: find orders where `revenue_usd` != `quantity * unit_price_usd` (allow a small rounding tolerance if needed).


In [ ]:
%%sql
SELECT
    order_id,
    customer_id,
    product_id,
    order_date,
    quantity,
    unit_price_usd,
    revenue_usd,
    (quantity * unit_price_usd) AS computed_revenue,
    ABS(revenue_usd - (quantity * unit_price_usd)) AS diff
FROM orders
WHERE revenue_usd IS NOT NULL
    AND quantity IS NOT NULL
    AND unit_price_usd IS NOT NULL
    AND ABS(revenue_usd - (quantity * unit_price_usd)) > 0.01
ORDER BY diff DESC

29. Produce a small order data-quality report: total rows, missing customer_id, missing order_date, non-positive quantity, non-positive unit price.


In [ ]:
%%sql
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS missing_customer_id,
    SUM(CASE WHEN order_date IS NULL THEN 1 ELSE 0 END) AS missing_order_date,
    SUM(CASE WHEN quantity <= 0 OR quantity IS NULL THEN 1 ELSE 0 END) AS non_positive_or_missing_quantity,
    SUM(CASE WHEN unit_price_usd <= 0 OR unit_price_usd IS NULL THEN 1 ELSE 0 END) AS non_positive_or_missing_unit_price
FROM orders



---

## Optional Challenge 


30. Customer lifetime value by region: compute each customer’s total revenue, then report average customer revenue by region.


In [ ]:
%%sql
WITH customer_totals AS (
    SELECT
        c.customer_id,
        c.region,
        SUM(o.revenue_usd) AS total_revenue
    FROM customers c
    JOIN orders o
        ON c.customer_id = o.customer_id
    GROUP BY c.customer_id, c.region
)

SELECT
    region,
    COUNT(*) AS customers_with_orders,
    ROUND(AVG(total_revenue), 2) AS avg_customer_revenue
FROM customer_totals
GROUP BY region
ORDER BY avg_customer_revenue DESC
